In [9]:
import pandas as pd
df = pd.read_csv(r"C:\Users\Lenovo\Desktop\Mobile Sales project\Mobile_Sales_Data.csv")

In [10]:
df.head(10)

,Transaction ID,Day,Month,Year,Day Name,Brand,Units Sold,Price Per Unit,Customer Name,Customer Age,City,Payment Method,Customer Ratings,Mobile Model
0,1,9,10,2021,Sat,Xiaomi,6,10174.70,Lalita Ahuja,38,Ludhiana,UPI,5,Redmi Note 10
1,2,9,10,2021,Saturday,Vivo,6,10565.19,Sneha Sharma,37,Delhi,Credit Card,3,Vivo Y51
2,3,9,10,2021,Saturday,Vivo,8,58527.58,Radha Srivastava,40,Mumbai,UPI,5,Vivo S1
3,4,10,10,2021,Sunday,Xiaomi,5,25563.98,Bhavana Arora,21,Mumbai,Credit Card,4,Mi 11
4,5,10,10,2021,Sunday,OnePlus,3,48168.02,Sneha Mehta,38,Gorakhpur,Cash,5,OnePlus 9
5,6,10,10,2021,Sunday,Samsung,3,37252.87,Reena Mehta,38,Jodhpur,UPI,3,Galaxy Note 20
6,7,10,10,2021,Sun,OnePlus,6,61805.80,Pankaj Alva,18,Delhi,Cash,4,OnePlus Nord
7,8,11,10,2021,Monday,Apple,2,65017.67,Mala Rawat,32,Vadodara,Cash,3,iPhone 12
8,9,11,10,2021,Mon,Vivo,6,69808.46,Pankaj Varma,34,Delhi,UPI,5,Vivo Y51
9,10,11,10,2021,Monday,Samsung,6,19221.05,Yogesh Sharma,31,Mumbai,Debit Card,2,Galaxy A51


In [20]:
import sys
!{sys.executable} -m pip install pymysql
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text

DB_USER = "root"
DB_PASS = quote_plus("@pswd1234")
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "mobile_sales_db"
SOURCE_FILE = r"C:\Users\Lenovo\Desktop\Mobile Sales project\Mobile_Sales_Data.csv"   



root_engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}")
with root_engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}"))
    conn.commit()

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# ---------- 3. READ ----------
if SOURCE_FILE.endswith(".csv"):
    df = pd.read_csv(SOURCE_FILE)
else:
    df = pd.read_excel(SOURCE_FILE)

print("Rows read:", len(df))

# ---------- 4. CLEAN ----------
# Standardise column names: "Customer Age" -> customer_age
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# Map likely variants to the names used in the SQL file
rename_map = {
    "customer_ratings": "customer_rating",
    "rating": "customer_rating",
    "mobile_name": "mobile_model",
    "model": "mobile_model",
    "payment_mode": "payment_method",
    "quantity": "units_sold",
    "unit_price": "price_per_unit",
    "price": "price_per_unit",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

df = df.drop_duplicates()

# Your file has separate Day / Month / Year columns (no single date column),
# so build a real date from them. Source "day" is the day of month.
df = df.rename(columns={"day": "day_of_month"})
df["date"] = pd.to_datetime(
    dict(year=df["year"], month=df["month"], day=df["day_of_month"]), errors="coerce"
)
df = df.dropna(subset=["date"])

# Text columns: trim whitespace
for col in ["brand", "mobile_model", "city", "payment_method", "customer_name"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# Numeric columns
for col in ["customer_age", "customer_rating", "units_sold", "price_per_unit", "total_sales"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Derive Total Sales if it is missing
if "total_sales" not in df.columns:
    df["total_sales"] = df["units_sold"] * df["price_per_unit"]

# Day name is recomputed from the date (source mixes "Sat" and "Saturday")
df["day"] = df["date"].dt.day_name()
df["month_no"] = df["date"].dt.month
df["month_name"] = df["date"].dt.month_name()

# Keep only the columns the table expects
cols = [
    "transaction_id", "date", "day", "month_no", "month_name",
    "customer_name", "customer_age", "city", "brand", "mobile_model",
    "payment_method", "customer_rating", "units_sold",
    "price_per_unit", "total_sales",
]
df = df[[c for c in cols if c in df.columns]]

# ---------- 5. LOAD ----------
create_sql = """
CREATE TABLE IF NOT EXISTS sales_data (
    transaction_id  INT PRIMARY KEY,
    date            DATE,
    day             VARCHAR(15),
    month_no        TINYINT,
    month_name      VARCHAR(15),
    customer_name   VARCHAR(100),
    customer_age    INT,
    city            VARCHAR(60),
    brand           VARCHAR(40),
    mobile_model    VARCHAR(60),
    payment_method  VARCHAR(30),
    customer_rating TINYINT,
    units_sold      INT,
    price_per_unit  DECIMAL(12,2),
    total_sales     DECIMAL(14,2),
    INDEX idx_date (date),
    INDEX idx_brand (brand),
    INDEX idx_city (city)
);
"""
with engine.connect() as conn:
    conn.execute(text(create_sql))
    conn.execute(text("TRUNCATE TABLE sales_data"))   # reload cleanly each run
    conn.commit()

df.to_sql("sales_data", engine, if_exists="append", index=False, chunksize=1000)
print(f"Loaded {len(df)} rows into {DB_NAME}.sales_data")

# ---------- 6. QUICK CHECK ----------
with engine.connect() as conn:
    result = conn.execute(text(
        "SELECT SUM(total_sales) AS total_sales, SUM(units_sold) AS qty, "
        "COUNT(*) AS transactions FROM sales_data"
    )).fetchone()
    print("Total Sales:", result[0], "| Quantity:", result[1], "| Transactions:", result[2])


Rows read: 3835
Loaded 3835 rows into mobile_sales_db.sales_data
Total Sales: 769204987.97 | Quantity: 19150 | Transactions: 3835
